
# Bot de Trading Unificado  
## Estrategia Donchian + Random Forest + Gestión de Riesgo Dinámica

Este notebook combina:

- La **estrategia mejorada**: Donchian 30, SMA 200, CCI, Random Forest y features de contexto.
- El **RiskManager dinámico** del primer bot: drawdown trailing, pérdida diaria, buffer, modos de riesgo, lote dinámico, anti-sobreoperación y bloqueos.
- Una configuración centralizada para adaptar fácilmente el sistema a otras cuentas de fondeo.
- Informes visuales y un resumen final de cumplimiento de reglas.

**Archivo esperado:** `datos_US500.csv` subido en la carpeta izquierda de Google Colab.


In [ ]:

!pip install statsmodels -q

import warnings
warnings.filterwarnings("ignore")

from collections import defaultdict
from dataclasses import dataclass
from datetime import datetime
from typing import Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, roc_curve

print("Librerías cargadas correctamente.")



## 1. Configuración centralizada

Para adaptar el notebook a otra cuenta de fondeo, cambia únicamente el bloque `PROP_FIRM`.

La estrategia y el gestor de riesgo leen todos sus parámetros desde `CONFIG`.


In [ ]:

# ============================================================
# CONFIGURACIÓN DE LA CUENTA DE FONDEO
# ============================================================

PROP_FIRM = {
    "name": "GetLeveraged Turbo Trade",
    "initial_balance": 100_000,
    "profit_target_pct": 0.06,
    "daily_loss_limit_pct": 0.03,
    "trailing_drawdown_pct": 0.06,
    "consistency_rule_pct": 0.20,
    "min_trade_duration_minutes": 2,
    "max_leverage": 30,

    # Límites internos más conservadores que los oficiales
    "internal_daily_risk_max": 1_500,
    "daily_profit_ceiling": 2_000,
    "daily_profit_ceiling_enabled": True,
    "max_trades_per_day": 8,
    "max_trades_per_day_enabled": True,
}

STRATEGY = {
    "donchian_period": 30,
    "sma_period": 200,
    "cci_period": 14,
    "cci_threshold": 100,
    "atr_period": 14,

    # Configuración optimizada del notebook mejorado
    "sl_multiplier": 0.5,
    "tp_multiplier": 1.5,
    "lookforward_bars": 30,
    "max_bars_in_trade": 32,

    # Filtro ML
    "ml_threshold": 0.50,
    "rf_n_estimators": 100,
    "rf_max_depth": 4,
    "rf_min_samples_leaf": 8,
}

RISK = {
    # Riesgo monetario dinámico por operación
    "max_risk_per_trade": 350,
    "min_risk_per_trade": 100,
    "min_signal_score": 50,

    # Separación entre entradas
    "min_bars_between_entries": 2,
    "anti_consecutive_enabled": True,
    "max_positions": 1,

    # Horario de negociación
    "trading_start_hour": 8,
    "trading_end_hour": 22,
    "filter_us_open": False,
    "us_open_start": 15.50,
    "us_open_end": 15.75,
}

DATA = {
    "csv_path": "/content/datos_US500.csv",
    "train_end_date": "2025-09-30",
    "test_end_date": "2025-12-31",
    "symbol": "US500",
    "timeframe": "M15",
    "point_value_per_lot": 1.0,
}

CONFIG = {**PROP_FIRM, **STRATEGY, **RISK, **DATA}

FEATURES = [
    "velas_sobre_sma",
    "densidad_senales",
    "dist_max_60d",
    "es_lunes",
    "es_viernes",
    "hora_sin",
]

WR_BREAKEVEN = (
    CONFIG["sl_multiplier"]
    / (CONFIG["sl_multiplier"] + CONFIG["tp_multiplier"])
    * 100
)

print("=" * 72)
print("BOT UNIFICADO — ESTRATEGIA MEJORADA + RIESGO DINÁMICO")
print("=" * 72)
print(f"Cuenta: {CONFIG['name']} | Balance: ${CONFIG['initial_balance']:,.0f}")
print(f"Objetivo: {CONFIG['profit_target_pct']*100:.1f}%")
print(f"DD trailing: {CONFIG['trailing_drawdown_pct']*100:.1f}%")
print(f"Pérdida diaria oficial: {CONFIG['daily_loss_limit_pct']*100:.1f}%")
print(f"Límite interno diario: ${CONFIG['internal_daily_risk_max']:,.0f}")
print(
    f"SL/TP: {CONFIG['sl_multiplier']}xATR / "
    f"{CONFIG['tp_multiplier']}xATR | Breakeven: {WR_BREAKEVEN:.1f}%"
)
print("=" * 72)



## 2. Carga de datos e indicadores

Se calculan internamente SMA, Donchian, ATR, CCI y las seis features de contexto utilizadas por el Random Forest.


In [ ]:

def load_data(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)

    required = {"time", "open", "high", "low", "close"}
    missing = required.difference(df.columns)
    if missing:
        raise ValueError(f"Faltan columnas obligatorias: {sorted(missing)}")

    df["time"] = pd.to_datetime(df["time"], errors="coerce")
    df = df.dropna(subset=["time"]).sort_values("time").reset_index(drop=True)

    for col in ["open", "high", "low", "close"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna(subset=["open", "high", "low", "close"]).reset_index(drop=True)
    return df


def calculate_indicators(df: pd.DataFrame, config: dict) -> pd.DataFrame:
    df = df.copy()

    # Tendencia
    df["sma_200"] = df["close"].rolling(config["sma_period"]).mean()

    # Donchian con shift(1), evitando que la vela actual entre en el canal
    don_p = config["donchian_period"]
    df["donchian_upper"] = df["high"].rolling(don_p).max().shift(1)
    df["donchian_lower"] = df["low"].rolling(don_p).min().shift(1)

    # ATR de Wilder
    hl = df["high"] - df["low"]
    hc = (df["high"] - df["close"].shift(1)).abs()
    lc = (df["low"] - df["close"].shift(1)).abs()
    tr = pd.concat([hl, hc, lc], axis=1).max(axis=1)
    df["atr"] = tr.ewm(
        alpha=1 / config["atr_period"], adjust=False
    ).mean()

    # CCI
    tp = (df["high"] + df["low"] + df["close"]) / 3
    sma_tp = tp.rolling(config["cci_period"]).mean()
    mean_dev = tp.rolling(config["cci_period"]).apply(
        lambda x: np.abs(x - x.mean()).mean(), raw=True
    )
    df["cci"] = (tp - sma_tp) / (0.015 * mean_dev.replace(0, np.nan))

    # Señal técnica base
    df["senal_base"] = (
        (df["close"] > df["donchian_upper"])
        & (df["close"] > df["sma_200"])
        & (df["cci"] > config["cci_threshold"])
    ).astype(int)

    # Feature 1: duración de la tendencia sobre SMA
    sobre_sma = (df["close"] > df["sma_200"]).astype(int).to_numpy()
    racha = np.zeros(len(df))
    contador = 0
    for i, valor in enumerate(sobre_sma):
        contador = contador + 1 if valor == 1 else 0
        racha[i] = contador
    df["velas_sobre_sma"] = np.log1p(racha)

    # Feature 2: densidad de señales en aproximadamente 20 sesiones
    df["densidad_senales"] = df["senal_base"].rolling(520).sum()

    # Feature 3: distancia al máximo aproximado de 60 sesiones, en ATR
    df["max_60d"] = df["high"].rolling(60 * 26).max()
    df["dist_max_60d"] = (
        (df["close"] - df["max_60d"]) / df["atr"].replace(0, np.nan)
    )

    # Features de calendario
    df["dia_semana"] = df["time"].dt.dayofweek
    df["es_lunes"] = (df["dia_semana"] == 0).astype(int)
    df["es_viernes"] = (df["dia_semana"] == 4).astype(int)
    hora_decimal = df["time"].dt.hour + df["time"].dt.minute / 60
    df["hora_sin"] = np.sin(2 * np.pi * hora_decimal / 24)

    # Distancias de salida
    df["stop_loss_points"] = df["atr"] * config["sl_multiplier"]
    df["take_profit_points"] = df["atr"] * config["tp_multiplier"]

    return df.dropna().reset_index(drop=True)


df_raw = load_data(CONFIG["csv_path"])
df = calculate_indicators(df_raw, CONFIG)

print(f"Velas originales: {len(df_raw):,}")
print(f"Velas utilizables: {len(df):,}")
print(f"Periodo: {df['time'].min()} → {df['time'].max()}")
print(f"Señales técnicas: {df['senal_base'].sum():,}")



## 3. Etiquetado de señales

Una señal se etiqueta como ganadora cuando el precio toca el TP antes que el SL dentro del horizonte definido.  
Las velas ambiguas se resuelven mediante la proximidad del precio de apertura a ambos niveles.


In [ ]:

def generate_labels(df: pd.DataFrame, config: dict) -> pd.DataFrame:
    df = df.copy()
    look = config["lookforward_bars"]

    closes = df["close"].to_numpy()
    opens = df["open"].to_numpy()
    highs = df["high"].to_numpy()
    lows = df["low"].to_numpy()
    atrs = df["atr"].to_numpy()

    labels = np.zeros(len(df), dtype=int)
    ambiguas = 0

    for i in range(len(df) - look):
        entry = closes[i]
        tp = entry + atrs[i] * config["tp_multiplier"]
        sl = entry - atrs[i] * config["sl_multiplier"]

        for j in range(i + 1, i + look + 1):
            hit_tp = highs[j] >= tp
            hit_sl = lows[j] <= sl

            if hit_tp and hit_sl:
                ambiguas += 1
                labels[i] = int(abs(opens[j] - tp) < abs(opens[j] - sl))
                break
            if hit_tp:
                labels[i] = 1
                break
            if hit_sl:
                labels[i] = 0
                break

    df["label"] = labels
    print(f"Velas ambiguas resueltas: {ambiguas:,}")
    return df


df = generate_labels(df, CONFIG)
senales = df[df["senal_base"] == 1]
print(f"WR natural de señales: {senales['label'].mean()*100:.2f}%")
print(f"Breakeven matemático: {WR_BREAKEVEN:.2f}%")



## 4. División temporal y entrenamiento del Random Forest

El modelo se entrena **solo con las señales del periodo de entrenamiento**.  
El escalador también se ajusta exclusivamente con ese periodo para evitar fuga de información.


In [ ]:

train_end = pd.Timestamp(CONFIG["train_end_date"])
test_end = pd.Timestamp(CONFIG["test_end_date"])

df_train = df[df["time"] <= train_end].copy()
df_test = df[(df["time"] > train_end) & (df["time"] <= test_end)].copy()
df_forward = df[df["time"] > test_end].copy()

s_train = df_train[df_train["senal_base"] == 1].copy()
s_test = df_test[df_test["senal_base"] == 1].copy()
s_forward = df_forward[df_forward["senal_base"] == 1].copy()

if len(s_train) < 50 or s_train["label"].nunique() < 2:
    raise ValueError(
        "No hay suficientes señales de entrenamiento o solo existe una clase."
    )

scaler = StandardScaler()
X_train = scaler.fit_transform(s_train[FEATURES])
y_train = s_train["label"].astype(int)

model = RandomForestClassifier(
    n_estimators=CONFIG["rf_n_estimators"],
    max_depth=CONFIG["rf_max_depth"],
    min_samples_leaf=CONFIG["rf_min_samples_leaf"],
    random_state=42,
    class_weight="balanced",
    n_jobs=-1,
)
model.fit(X_train, y_train)

def safe_auc(signal_df: pd.DataFrame) -> float:
    if signal_df.empty or signal_df["label"].nunique() < 2:
        return np.nan
    X = scaler.transform(signal_df[FEATURES])
    return roc_auc_score(signal_df["label"], model.predict_proba(X)[:, 1])

auc_train = safe_auc(s_train)
auc_test = safe_auc(s_test)
auc_forward = safe_auc(s_forward)

print("División temporal:")
print(f"Train:   {df_train['time'].min().date()} → {df_train['time'].max().date()} | {len(s_train):,} señales")
print(f"Test:    {df_test['time'].min().date()} → {df_test['time'].max().date()} | {len(s_test):,} señales")
print(f"Forward: {df_forward['time'].min().date()} → {df_forward['time'].max().date()} | {len(s_forward):,} señales")
print()
print(f"AUC Train:   {auc_train:.4f}")
print(f"AUC Test:    {auc_test:.4f}" if not np.isnan(auc_test) else "AUC Test: N/D")
print(f"AUC Forward: {auc_forward:.4f}" if not np.isnan(auc_forward) else "AUC Forward: N/D")



## 5. Gestor de riesgo dinámico

Esta sección conserva la lógica del primer bot:

- drawdown trailing sobre la equity máxima;
- pérdida diaria oficial e interna;
- buffer dinámico;
- modos NORMAL, CONSERVADOR, SUPERVIVENCIA y BLOQUEADO;
- riesgo monetario y lote variables;
- límite de operaciones;
- techo de beneficio diario;
- filtro horario;
- anti-consecutivo;
- regla de consistencia.


In [ ]:

class ConsistencyTracker:
    def __init__(self, config: dict):
        self.threshold = config["consistency_rule_pct"]
        self.daily_profits = {}

    def update_day(self, date, pnl: float) -> None:
        if pnl > 0:
            self.daily_profits[date] = float(pnl)

    def summary(self) -> dict:
        if not self.daily_profits:
            return {
                "best_day_pnl": 0.0,
                "total_positive_days": 0.0,
                "consistency_pct": 0.0,
                "compliant": True,
            }

        total = sum(self.daily_profits.values())
        best = max(self.daily_profits.values())
        pct = best / total * 100 if total > 0 else 0

        return {
            "best_day_pnl": round(best, 2),
            "total_positive_days": round(total, 2),
            "consistency_pct": round(pct, 2),
            "compliant": pct <= self.threshold * 100,
        }


class AntiConsecutiveTracker:
    def __init__(self, config: dict):
        self.enabled = config["anti_consecutive_enabled"]
        self.min_bars = config["min_bars_between_entries"]
        self.last_entry_bar = None

    def can_enter(self, bar_index: int) -> Tuple[bool, str]:
        if not self.enabled or self.last_entry_bar is None:
            return True, "OK"

        elapsed = bar_index - self.last_entry_bar
        if elapsed < self.min_bars:
            return False, f"Anti-consecutivo: {elapsed}/{self.min_bars} velas"
        return True, "OK"

    def register(self, bar_index: int) -> None:
        self.last_entry_bar = bar_index

    def reset_day(self) -> None:
        self.last_entry_bar = None


class RiskManager:
    def __init__(self, config: dict):
        self.config = config

        self.initial_balance = float(config["initial_balance"])
        self.balance = self.initial_balance
        self.equity = self.initial_balance
        self.max_equity = self.initial_balance

        self.current_day = None
        self.day_start_balance = self.initial_balance
        self.daily_pnl = 0.0
        self.daily_risk_used = 0.0
        self.daily_trades = []

        self.blocked_today = False
        self.blocked_total = False
        self.block_reason = ""
        self.profit_target_reached = False
        self.ceiling_reached_today = False
        self.max_trades_reached_today = False

        self.trade_log = []
        self.daily_log = []
        self.rejection_log = defaultdict(int)

        self.consistency = ConsistencyTracker(config)
        self.anti_consecutive = AntiConsecutiveTracker(config)

    def update_day(self, timestamp: pd.Timestamp) -> None:
        day = timestamp.date()
        if self.current_day != day:
            if self.current_day is not None:
                self._close_day()

            self.current_day = day
            self.day_start_balance = self.balance
            self.daily_pnl = 0.0
            self.daily_risk_used = 0.0
            self.daily_trades = []
            self.blocked_today = False
            self.ceiling_reached_today = False
            self.max_trades_reached_today = False
            self.anti_consecutive.reset_day()

    def _close_day(self) -> None:
        summary = self._daily_summary()
        self.daily_log.append(summary)
        self.consistency.update_day(self.current_day, self.daily_pnl)

    def get_drawdown_floor(self) -> float:
        # Regla trailing: el suelo asciende con la equity máxima.
        # Tras alcanzar el objetivo, nunca cae por debajo del capital inicial.
        floor = self.max_equity * (1 - self.config["trailing_drawdown_pct"])
        if self.profit_target_reached:
            floor = max(floor, self.initial_balance)
        return floor

    def get_buffer_ratio(self) -> float:
        floor = self.get_drawdown_floor()
        total_buffer = self.max_equity - floor
        remaining = self.equity - floor

        if total_buffer <= 0:
            return 0.0
        return float(np.clip(remaining / total_buffer, 0, 1))

    def get_buffer_used_pct(self) -> float:
        return (1 - self.get_buffer_ratio()) * 100

    def get_risk_mode(self) -> str:
        buffer = self.get_buffer_ratio()
        if buffer > 0.70:
            return "NORMAL"
        if buffer > 0.40:
            return "CONSERVADOR"
        if buffer > 0.20:
            return "SUPERVIVENCIA"
        return "BLOQUEADO"

    def check_profit_target(self) -> None:
        target = self.initial_balance * (1 + self.config["profit_target_pct"])
        self.profit_target_reached = self.max_equity >= target

    def check_trailing_drawdown(self) -> Tuple[bool, str]:
        if self.equity <= self.get_drawdown_floor():
            self.blocked_total = True
            self.block_reason = "Drawdown trailing alcanzado"
            return False, self.block_reason
        return True, "OK"

    def check_daily_loss(self) -> Tuple[bool, str]:
        official = self.day_start_balance * self.config["daily_loss_limit_pct"]
        internal = self.config["internal_daily_risk_max"]

        if self.daily_pnl <= -official:
            self.blocked_today = True
            return False, "Pérdida diaria oficial alcanzada"

        if self.daily_pnl <= -internal:
            self.blocked_today = True
            return False, "Límite interno diario alcanzado"

        return True, "OK"

    def check_daily_ceiling(self) -> Tuple[bool, str]:
        if not self.config["daily_profit_ceiling_enabled"]:
            return True, "OK"

        if self.daily_pnl >= self.config["daily_profit_ceiling"]:
            self.ceiling_reached_today = True
            return False, "Techo de beneficio diario alcanzado"

        return True, "OK"

    def check_max_trades(self) -> Tuple[bool, str]:
        if not self.config["max_trades_per_day_enabled"]:
            return True, "OK"

        if len(self.daily_trades) >= self.config["max_trades_per_day"]:
            self.max_trades_reached_today = True
            return False, "Máximo de operaciones diarias alcanzado"

        return True, "OK"

    def check_hours(self, timestamp: pd.Timestamp) -> Tuple[bool, str]:
        hour_decimal = timestamp.hour + timestamp.minute / 60

        if not (
            self.config["trading_start_hour"]
            <= hour_decimal
            < self.config["trading_end_hour"]
        ):
            return False, "Fuera de horario"

        if self.config["filter_us_open"]:
            if (
                self.config["us_open_start"]
                <= hour_decimal
                < self.config["us_open_end"]
            ):
                return False, "Filtro apertura estadounidense"

        return True, "OK"

    def _daily_budget(self) -> float:
        budgets = {
            "NORMAL": min(1_000, self.config["internal_daily_risk_max"]),
            "CONSERVADOR": min(600, self.config["internal_daily_risk_max"]),
            "SUPERVIVENCIA": min(250, self.config["internal_daily_risk_max"]),
            "BLOQUEADO": 0,
        }

        budget = budgets[self.get_risk_mode()]

        if self.config["daily_profit_ceiling_enabled"]:
            remaining_to_ceiling = (
                self.config["daily_profit_ceiling"] - self.daily_pnl
            )
            budget = min(budget, max(0, remaining_to_ceiling * 0.5))

        return max(0.0, budget)

    def calculate_risk(self, signal_score: float) -> float:
        if signal_score < self.config["min_signal_score"]:
            return 0.0

        buffer = self.get_buffer_ratio()
        if buffer <= 0.20:
            return 0.0

        min_risk = self.config["min_risk_per_trade"]
        max_risk = self.config["max_risk_per_trade"]

        scale = (buffer - 0.20) / 0.80
        base = min_risk + (max_risk - min_risk) * scale

        if signal_score >= 80:
            quality_mult = 1.00
        elif signal_score >= 65:
            quality_mult = 0.75
        else:
            quality_mult = 0.50

        remaining_budget = self._daily_budget() - self.daily_risk_used
        risk = min(base * quality_mult, remaining_budget)

        return round(risk, 2) if risk >= min_risk else 0.0

    def calculate_lot(self, risk_usd: float, sl_points: float) -> float:
        denominator = sl_points * self.config["point_value_per_lot"]
        if risk_usd <= 0 or denominator <= 0:
            return 0.0
        return round(risk_usd / denominator, 4)

    def evaluate_trade(self, request: dict, bar_index: int) -> dict:
        self.update_day(request["timestamp"])

        base = {
            "allowed": False,
            "reason": "",
            "risk_usd": 0.0,
            "lot_size": 0.0,
            "mode": self.get_risk_mode(),
        }

        if self.blocked_total:
            return {**base, "reason": "Cuenta bloqueada"}

        if self.blocked_today:
            return {**base, "reason": "Bloqueado durante el día"}

        checks = [
            self.check_hours(request["timestamp"]),
            self.check_trailing_drawdown(),
            self.check_daily_loss(),
            self.check_daily_ceiling(),
            self.check_max_trades(),
            self.anti_consecutive.can_enter(bar_index),
        ]

        for ok, reason in checks:
            if not ok:
                self.rejection_log[reason] += 1
                return {**base, "reason": reason}

        risk_usd = self.calculate_risk(request["signal_score"])
        if risk_usd <= 0:
            reason = "Riesgo cero por score o buffer"
            self.rejection_log[reason] += 1
            return {**base, "reason": reason}

        lot = self.calculate_lot(risk_usd, request["stop_loss_points"])
        if lot <= 0:
            reason = "Lote inválido"
            self.rejection_log[reason] += 1
            return {**base, "reason": reason}

        return {
            "allowed": True,
            "reason": "Autorizada",
            "risk_usd": risk_usd,
            "lot_size": lot,
            "mode": self.get_risk_mode(),
        }

    def confirm_open(self, risk_usd: float, bar_index: int) -> None:
        self.daily_risk_used += risk_usd
        self.anti_consecutive.register(bar_index)

    def register_trade(
        self,
        open_time,
        close_time,
        pnl: float,
        details: dict,
    ) -> None:
        self.balance += pnl
        self.equity = self.balance
        self.max_equity = max(self.max_equity, self.equity)
        self.daily_pnl += pnl

        self.check_profit_target()

        record = {
            "open_time": open_time,
            "close_time": close_time,
            "pnl": round(pnl, 2),
            "balance": round(self.balance, 2),
            "equity": round(self.equity, 2),
            "equity_max": round(self.max_equity, 2),
            "drawdown_floor": round(self.get_drawdown_floor(), 2),
            "risk_mode": self.get_risk_mode(),
            **details,
        }

        self.trade_log.append(record)
        self.daily_trades.append(record)

        self.check_trailing_drawdown()
        self.check_daily_loss()
        self.check_daily_ceiling()

    def _daily_summary(self) -> dict:
        total = len(self.daily_trades)
        wins = sum(t["pnl"] > 0 for t in self.daily_trades)

        return {
            "date": self.current_day,
            "start_balance": round(self.day_start_balance, 2),
            "end_balance": round(self.balance, 2),
            "daily_pnl": round(self.daily_pnl, 2),
            "trades": total,
            "wins": wins,
            "win_rate": round(wins / total * 100, 2) if total else 0.0,
            "equity_max": round(self.max_equity, 2),
            "drawdown_floor": round(self.get_drawdown_floor(), 2),
            "distance_to_floor": round(self.equity - self.get_drawdown_floor(), 2),
            "buffer_used_pct": round(self.get_buffer_used_pct(), 2),
            "risk_mode": self.get_risk_mode(),
            "blocked_today": self.blocked_today,
        }

    def final_metrics(self) -> dict:
        if self.current_day is not None:
            self._close_day()
            self.current_day = None

        trades = pd.DataFrame(self.trade_log)
        daily = pd.DataFrame(self.daily_log)

        total = len(trades)
        wins = int((trades["pnl"] > 0).sum()) if total else 0
        gains = trades.loc[trades["pnl"] > 0, "pnl"].sum() if total else 0
        losses = abs(trades.loc[trades["pnl"] < 0, "pnl"].sum()) if total else 0
        profit_factor = gains / losses if losses > 0 else np.inf if gains > 0 else 0

        worst_day = daily["daily_pnl"].min() if not daily.empty else 0
        max_daily_loss_allowed = (
            daily["start_balance"] * self.config["daily_loss_limit_pct"]
        ).max() if not daily.empty else self.initial_balance * self.config["daily_loss_limit_pct"]

        consistency = self.consistency.summary()

        return {
            "initial_balance": self.initial_balance,
            "final_balance": round(self.balance, 2),
            "return_pct": round((self.balance / self.initial_balance - 1) * 100, 2),
            "total_pnl": round(self.balance - self.initial_balance, 2),
            "max_equity": round(self.max_equity, 2),
            "drawdown_floor": round(self.get_drawdown_floor(), 2),
            "distance_to_floor": round(self.equity - self.get_drawdown_floor(), 2),
            "total_trades": total,
            "wins": wins,
            "win_rate": round(wins / total * 100, 2) if total else 0,
            "profit_factor": round(profit_factor, 3) if np.isfinite(profit_factor) else np.inf,
            "profit_target_reached": self.profit_target_reached,
            "trailing_dd_compliant": not self.blocked_total,
            "daily_loss_compliant": abs(min(0, worst_day)) < max_daily_loss_allowed,
            "internal_daily_limit_compliant": abs(min(0, worst_day)) <= self.config["internal_daily_risk_max"],
            "consistency": consistency,
            "worst_day": round(worst_day, 2),
            "rejections": dict(self.rejection_log),
        }



## 6. Generador de señales y orquestador

El Random Forest filtra las señales técnicas.  
El `RiskManager` mantiene siempre la última palabra sobre la apertura y el tamaño de la posición.


In [ ]:

class ImprovedSignalGenerator:
    def __init__(
        self,
        config: dict,
        model: RandomForestClassifier,
        scaler: StandardScaler,
    ):
        self.config = config
        self.model = model
        self.scaler = scaler

    def evaluate(self, row: pd.Series) -> dict:
        result = {
            "signal": 0,
            "signal_score": 0,
            "ml_probability": 0.0,
            "entry_price": float(row["close"]),
            "stop_loss_points": float(row["stop_loss_points"]),
            "take_profit_points": float(row["take_profit_points"]),
        }

        if int(row["senal_base"]) != 1:
            return result

        X = pd.DataFrame([row[FEATURES].to_dict()], columns=FEATURES)
        probability = float(
            self.model.predict_proba(self.scaler.transform(X))[0, 1]
        )

        if probability < self.config["ml_threshold"]:
            return result

        result["signal"] = 1
        result["ml_probability"] = probability
        result["signal_score"] = int(np.clip(probability * 100, 0, 100))
        return result


class TradingOrchestrator:
    def __init__(
        self,
        config: dict,
        risk_manager: RiskManager,
        signal_generator: ImprovedSignalGenerator,
    ):
        self.config = config
        self.rm = risk_manager
        self.sg = signal_generator
        self.position = None
        self.execution_log = []

    def process_bar(self, row: pd.Series, bar_index: int) -> None:
        self.rm.update_day(row["time"])

        # Primero se gestiona la posición ya abierta.
        if self.position is not None:
            self._check_exit(row, bar_index)

        # No se abre una segunda posición en la misma vela tras cerrar.
        if self.position is None:
            self._evaluate_entry(row, bar_index)

    def _evaluate_entry(self, row: pd.Series, bar_index: int) -> None:
        signal = self.sg.evaluate(row)
        if signal["signal"] == 0:
            return

        request = {
            "timestamp": row["time"],
            "signal_score": signal["signal_score"],
            "entry_price": signal["entry_price"],
            "stop_loss_points": signal["stop_loss_points"],
        }

        decision = self.rm.evaluate_trade(request, bar_index)
        if not decision["allowed"]:
            return

        entry = signal["entry_price"]

        self.position = {
            "open_time": row["time"],
            "open_bar": bar_index,
            "entry_price": entry,
            "stop_loss": entry - signal["stop_loss_points"],
            "take_profit": entry + signal["take_profit_points"],
            "lot_size": decision["lot_size"],
            "risk_usd": decision["risk_usd"],
            "signal_score": signal["signal_score"],
            "ml_probability": signal["ml_probability"],
        }

        self.rm.confirm_open(decision["risk_usd"], bar_index)

    def _check_exit(self, row: pd.Series, bar_index: int) -> None:
        pos = self.position
        if pos is None:
            return

        exit_price = None
        exit_reason = None

        hit_tp = row["high"] >= pos["take_profit"]
        hit_sl = row["low"] <= pos["stop_loss"]

        # Vela ambigua: resolución coherente con el etiquetado.
        if hit_tp and hit_sl:
            if abs(row["open"] - pos["take_profit"]) < abs(
                row["open"] - pos["stop_loss"]
            ):
                exit_price = pos["take_profit"]
                exit_reason = "TP_AMBIGUA"
            else:
                exit_price = pos["stop_loss"]
                exit_reason = "SL_AMBIGUA"
        elif hit_tp:
            exit_price = pos["take_profit"]
            exit_reason = "TP"
        elif hit_sl:
            exit_price = pos["stop_loss"]
            exit_reason = "SL"

        bars_in_trade = bar_index - pos["open_bar"]

        if (
            exit_price is None
            and bars_in_trade >= self.config["max_bars_in_trade"]
        ):
            exit_price = float(row["close"])
            exit_reason = "TIEMPO"

        if exit_price is None:
            return

        pnl = (
            (exit_price - pos["entry_price"])
            * pos["lot_size"]
            * self.config["point_value_per_lot"]
        )

        self.rm.register_trade(
            pos["open_time"],
            row["time"],
            pnl,
            {
                "entry_price": pos["entry_price"],
                "exit_price": exit_price,
                "exit_reason": exit_reason,
                "lot_size": pos["lot_size"],
                "risk_usd": pos["risk_usd"],
                "signal_score": pos["signal_score"],
                "ml_probability": pos["ml_probability"],
                "bars_in_trade": bars_in_trade,
            },
        )

        self.execution_log.append(self.rm.trade_log[-1])
        self.position = None


def run_backtest(
    period_df: pd.DataFrame,
    config: dict,
    model,
    scaler,
    period_name: str,
) -> dict:
    rm = RiskManager(config)
    sg = ImprovedSignalGenerator(config, model, scaler)
    orchestrator = TradingOrchestrator(config, rm, sg)

    for bar_index, (_, row) in enumerate(period_df.reset_index(drop=True).iterrows()):
        orchestrator.process_bar(row, bar_index)
        if rm.blocked_total:
            break

    # Cierre de posición pendiente al final del periodo.
    if orchestrator.position is not None and len(period_df) > 0:
        last_row = period_df.iloc[-1]
        pos = orchestrator.position
        exit_price = float(last_row["close"])
        pnl = (
            (exit_price - pos["entry_price"])
            * pos["lot_size"]
            * config["point_value_per_lot"]
        )
        rm.register_trade(
            pos["open_time"],
            last_row["time"],
            pnl,
            {
                "entry_price": pos["entry_price"],
                "exit_price": exit_price,
                "exit_reason": "FIN_PERIODO",
                "lot_size": pos["lot_size"],
                "risk_usd": pos["risk_usd"],
                "signal_score": pos["signal_score"],
                "ml_probability": pos["ml_probability"],
                "bars_in_trade": len(period_df) - 1 - pos["open_bar"],
            },
        )

    metrics = rm.final_metrics()

    return {
        "period": period_name,
        "metrics": metrics,
        "trades": pd.DataFrame(rm.trade_log),
        "daily": pd.DataFrame(rm.daily_log),
        "rejections": pd.Series(rm.rejection_log, dtype=float),
    }



## 7. Ejecución del backtest y forward test


In [ ]:

results = {}

if not df_test.empty:
    results["TEST"] = run_backtest(
        df_test, CONFIG, model, scaler, "TEST"
    )

if not df_forward.empty:
    results["FORWARD"] = run_backtest(
        df_forward, CONFIG, model, scaler, "FORWARD"
    )

for period, result in results.items():
    m = result["metrics"]
    print(
        f"{period:<8} | Balance ${m['final_balance']:,.2f} | "
        f"Rentabilidad {m['return_pct']:+.2f}% | "
        f"Ops {m['total_trades']} | WR {m['win_rate']:.1f}% | "
        f"PF {m['profit_factor']}"
    )



## 8. Resumen final de cumplimiento de reglas

El informe separa claramente:

- cumplimiento de reglas críticas;
- cumplimiento de límites internos;
- objetivo de rentabilidad;
- consistencia;
- motivos de rechazo del gestor.


In [ ]:

def yes_no(value: bool) -> str:
    return "✅ SÍ" if value else "❌ NO"


def print_compliance_report(period: str, result: dict, config: dict) -> None:
    m = result["metrics"]
    c = m["consistency"]

    target_usd = config["initial_balance"] * config["profit_target_pct"]
    official_daily_limit = (
        config["initial_balance"] * config["daily_loss_limit_pct"]
    )

    print("\n" + "=" * 78)
    print(f"RESUMEN FINAL DE CUMPLIMIENTO — {period}")
    print(f"{config['name']} | {config['symbol']} {config['timeframe']}")
    print("=" * 78)

    print("\nRENDIMIENTO")
    print(f"Balance inicial:             ${m['initial_balance']:,.2f}")
    print(f"Balance final:               ${m['final_balance']:,.2f}")
    print(f"PnL total:                   ${m['total_pnl']:,.2f}")
    print(f"Rentabilidad:                {m['return_pct']:+.2f}%")
    print(f"Objetivo monetario:          ${target_usd:,.2f}")
    print(f"Objetivo alcanzado:          {yes_no(m['profit_target_reached'])}")

    print("\nOPERACIONES")
    print(f"Total operaciones:           {m['total_trades']}")
    print(f"Ganadoras:                   {m['wins']}")
    print(f"Win rate:                    {m['win_rate']:.2f}%")
    print(f"Breakeven teórico:           {WR_BREAKEVEN:.2f}%")
    print(f"Profit factor:               {m['profit_factor']}")

    print("\nREGLAS DE FONDEO")
    print(
        f"Drawdown trailing "
        f"({config['trailing_drawdown_pct']*100:.1f}%): "
        f"{yes_no(m['trailing_dd_compliant'])}"
    )
    print(
        f"Pérdida diaria oficial "
        f"(${official_daily_limit:,.0f} aprox.): "
        f"{yes_no(m['daily_loss_compliant'])}"
    )
    print(
        f"Límite interno diario "
        f"(${config['internal_daily_risk_max']:,.0f}): "
        f"{yes_no(m['internal_daily_limit_compliant'])}"
    )
    print(
        f"Consistencia "
        f"(máx. {config['consistency_rule_pct']*100:.0f}%): "
        f"{yes_no(c['compliant'])}"
    )
    print(f"Score de consistencia:       {c['consistency_pct']:.2f}%")
    print(f"Peor día:                    ${m['worst_day']:,.2f}")
    print(f"Equity máxima:               ${m['max_equity']:,.2f}")
    print(f"Floor actual:                ${m['drawdown_floor']:,.2f}")
    print(f"Distancia al floor:          ${m['distance_to_floor']:,.2f}")

    critical_rules = [
        m["trailing_dd_compliant"],
        m["daily_loss_compliant"],
    ]

    print("\nCONCLUSIÓN")
    if all(critical_rules):
        print("✅ El bot respeta las reglas críticas de la cuenta en este periodo.")
    else:
        print("⛔ El bot incumple al menos una regla crítica en este periodo.")

    if all(critical_rules) and not m["profit_target_reached"]:
        print("⚠️ Protege la cuenta, pero todavía no alcanza el objetivo de rentabilidad.")

    print("\nRECHAZOS DEL RISKMANAGER")
    if m["rejections"]:
        for reason, count in sorted(
            m["rejections"].items(), key=lambda item: item[1], reverse=True
        ):
            print(f"- {reason}: {count}")
    else:
        print("- No se registraron rechazos.")


for period, result in results.items():
    print_compliance_report(period, result, CONFIG)



## 9. Panel visual

El panel presenta:

- curva de equity;
- PnL diario;
- uso del buffer de drawdown;
- motivos de rechazo;
- importancia de variables;
- comparación de métricas entre test y forward.


In [ ]:

def plot_dashboard(results: dict, model, features: list, config: dict) -> None:
    if not results:
        print("No hay resultados para representar.")
        return

    fig = plt.figure(figsize=(18, 16))
    gs = gridspec.GridSpec(4, 2, figure=fig, hspace=0.38, wspace=0.25)

    # 1. Equity
    ax1 = fig.add_subplot(gs[0, :])
    for period, result in results.items():
        trades = result["trades"]
        if not trades.empty:
            ax1.plot(
                trades["close_time"],
                trades["balance"],
                linewidth=1.7,
                label=period,
            )
    ax1.axhline(
        config["initial_balance"],
        linestyle="--",
        linewidth=1,
        label="Balance inicial",
    )
    ax1.axhline(
        config["initial_balance"] * (1 + config["profit_target_pct"]),
        linestyle=":",
        linewidth=1.5,
        label="Objetivo",
    )
    ax1.set_title("Curva de equity")
    ax1.set_ylabel("Balance ($)")
    ax1.legend()
    ax1.grid(alpha=0.3)

    # 2. PnL diario
    ax2 = fig.add_subplot(gs[1, 0])
    for period, result in results.items():
        daily = result["daily"]
        if not daily.empty:
            ax2.plot(
                pd.to_datetime(daily["date"]),
                daily["daily_pnl"],
                marker=".",
                linewidth=1,
                label=period,
            )
    ax2.axhline(0, linewidth=0.8)
    ax2.axhline(
        -config["internal_daily_risk_max"],
        linestyle="--",
        linewidth=1,
        label="Límite interno",
    )
    ax2.set_title("PnL diario")
    ax2.set_ylabel("$")
    ax2.legend()
    ax2.grid(alpha=0.3)

    # 3. Buffer
    ax3 = fig.add_subplot(gs[1, 1])
    for period, result in results.items():
        daily = result["daily"]
        if not daily.empty:
            ax3.plot(
                pd.to_datetime(daily["date"]),
                daily["buffer_used_pct"],
                linewidth=1.3,
                label=period,
            )
    ax3.axhline(50, linestyle="--", linewidth=1, label="Aviso 50%")
    ax3.axhline(75, linestyle="--", linewidth=1, label="Crítico 75%")
    ax3.set_ylim(0, 100)
    ax3.set_title("Buffer de drawdown utilizado")
    ax3.set_ylabel("%")
    ax3.legend()
    ax3.grid(alpha=0.3)

    # 4. Rechazos
    ax4 = fig.add_subplot(gs[2, 0])
    all_rejections = defaultdict(int)
    for result in results.values():
        for reason, count in result["metrics"]["rejections"].items():
            all_rejections[reason] += count

    if all_rejections:
        rej = pd.Series(all_rejections).sort_values()
        ax4.barh(rej.index, rej.values)
    ax4.set_title("Señales rechazadas por el RiskManager")
    ax4.set_xlabel("Cantidad")
    ax4.grid(axis="x", alpha=0.3)

    # 5. Importancia de features
    ax5 = fig.add_subplot(gs[2, 1])
    importance = pd.Series(
        model.feature_importances_, index=features
    ).sort_values()
    ax5.barh(importance.index, importance.values)
    ax5.set_title("Importancia de variables — Random Forest")
    ax5.set_xlabel("Importancia relativa")
    ax5.grid(axis="x", alpha=0.3)

    # 6. Comparativa de rentabilidad
    ax6 = fig.add_subplot(gs[3, 0])
    periods = list(results)
    returns = [results[p]["metrics"]["return_pct"] for p in periods]
    ax6.bar(periods, returns)
    ax6.axhline(
        config["profit_target_pct"] * 100,
        linestyle="--",
        linewidth=1.5,
        label="Objetivo",
    )
    ax6.set_title("Rentabilidad por periodo")
    ax6.set_ylabel("%")
    ax6.legend()
    ax6.grid(axis="y", alpha=0.3)

    # 7. Win rate vs breakeven
    ax7 = fig.add_subplot(gs[3, 1])
    wrs = [results[p]["metrics"]["win_rate"] for p in periods]
    ax7.bar(periods, wrs)
    ax7.axhline(
        WR_BREAKEVEN,
        linestyle="--",
        linewidth=1.5,
        label=f"Breakeven {WR_BREAKEVEN:.1f}%",
    )
    ax7.set_title("Win rate frente a breakeven")
    ax7.set_ylabel("%")
    ax7.legend()
    ax7.grid(axis="y", alpha=0.3)

    plt.suptitle(
        "BOT UNIFICADO — ESTRATEGIA ML + GESTIÓN DE RIESGO DINÁMICA",
        fontsize=14,
        fontweight="bold",
    )
    plt.show()


plot_dashboard(results, model, FEATURES, CONFIG)



## 10. Exportación de resultados

Se guardan las operaciones y los resúmenes diarios de cada periodo para su revisión.


In [ ]:

for period, result in results.items():
    result["trades"].to_csv(
        f"operaciones_{period.lower()}.csv", index=False
    )
    result["daily"].to_csv(
        f"resumen_diario_{period.lower()}.csv", index=False
    )

print("Archivos CSV exportados correctamente.")
